In [ ]:
import pickle, os

# ── Load models ────────────────────────────────────────────────────────────
def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

bat_model    = load_pkl("bat_model.pkl")
ridge_bat    = load_pkl("ridge_bat.pkl")
lasso_bat    = load_pkl("lasso_bat.pkl")
elastic_bat  = load_pkl("elastic_bat.pkl")
rf_bat       = load_pkl("rf_bat.pkl")

bowl_model   = load_pkl("bowl_model.pkl")
ridge_bowl   = load_pkl("ridge_bowl.pkl")
lasso_bowl   = load_pkl("lasso_bowl.pkl")
elastic_bowl = load_pkl("elastic_bowl.pkl")
rf_bowl      = load_pkl("rf_bowl.pkl")

print("All models loaded ✓")

All models loaded ✓


In [ ]:
import pandas as pd

batting_clean = pd.read_csv("batting_clean.csv")
bowling_clean = pd.read_csv("bowling_clean.csv")

print(f"Batting: {batting_clean.shape}, Bowling: {bowling_clean.shape} ✓")

Batting: (16515, 11), Bowling: (12978, 17) ✓


In [ ]:
# =========================
# IMPORTS
# =========================
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle

from sklearn.metrics import r2_score, mean_absolute_error

# =========================
# LOAD DATA
# =========================
batting_clean = pd.read_csv("batting_clean.csv")
bowling_clean = pd.read_csv("bowling_clean.csv")

# Feature engineering
for df in [batting_clean, bowling_clean]:
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["match_year"] = df["date"].dt.year
        df["match_month"] = df["date"].dt.month
        df["match_weekday"] = df["date"].dt.weekday
        df.drop(columns=["date"], inplace=True, errors="ignore")

    if "season_year" not in df.columns:
        df["season_year"] = df["season"].astype(str).str[:4].astype(int)

# =========================
# LOAD MODELS
# =========================
bat_linear = pickle.load(open("bat_model.pkl", "rb"))
bat_ridge = pickle.load(open("ridge_bat.pkl", "rb"))
bat_lasso = pickle.load(open("lasso_bat.pkl", "rb"))
bat_elastic = pickle.load(open("elastic_bat.pkl", "rb"))
bat_rf = pickle.load(open("rf_bat.pkl", "rb"))

bowl_linear = pickle.load(open("bowl_model.pkl", "rb"))
bowl_ridge = pickle.load(open("ridge_bowl.pkl", "rb"))
bowl_lasso = pickle.load(open("lasso_bowl.pkl", "rb"))
bowl_elastic = pickle.load(open("elastic_bowl.pkl", "rb"))
bowl_rf = pickle.load(open("rf_bowl.pkl", "rb"))

# =========================
# LOAD FEATURES
# =========================
bat_features = pickle.load(open("bat_features.pkl", "rb"))
bowl_features = pickle.load(open("bowl_features.pkl", "rb"))

# =========================
# HELPER FUNCTIONS
# =========================
def align_features(df, features):
    for col in features:
        if col not in df.columns:
            df[col] = np.nan
    return df[features]

# =========================
# TEST SETS
# =========================
test_bat = batting_clean[batting_clean["season_year"] >= 2023]
test_bowl = bowling_clean[bowling_clean["season_year"] >= 2023]

X_test_bat = align_features(test_bat.drop(columns=["runs"]), bat_features)
y_test_bat = test_bat["runs"]

X_test_bowl = align_features(test_bowl.drop(columns=["economy"]), bowl_features)
y_test_bowl = test_bowl["economy"]

# =========================
# DASHBOARD FUNCTIONS
# =========================

def show_model_performance():
    models = ["Linear","Ridge","Lasso","Elastic","RF"]

    bat_scores = [
        r2_score(y_test_bat, bat_linear.predict(X_test_bat)),
        r2_score(y_test_bat, bat_ridge.predict(X_test_bat)),
        r2_score(y_test_bat, bat_lasso.predict(X_test_bat)),
        r2_score(y_test_bat, bat_elastic.predict(X_test_bat)),
        r2_score(y_test_bat, bat_rf.predict(X_test_bat)),
    ]

    bowl_scores = [
        r2_score(y_test_bowl, bowl_linear.predict(X_test_bowl)),
        r2_score(y_test_bowl, bowl_ridge.predict(X_test_bowl)),
        r2_score(y_test_bowl, bowl_lasso.predict(X_test_bowl)),
        r2_score(y_test_bowl, bowl_elastic.predict(X_test_bowl)),
        r2_score(y_test_bowl, bowl_rf.predict(X_test_bowl)),
    ]

    fig, ax = plt.subplots(1,2, figsize=(12,5))
    ax[0].bar(models, bat_scores)
    ax[0].set_title("Batting R²")

    ax[1].bar(models, bowl_scores)
    ax[1].set_title("Bowling R²")

    plt.show()

    print("Best Batting:", models[np.argmax(bat_scores)])
    print("Best Bowling:", models[np.argmax(bowl_scores)])


def show_actual_vs_predicted():
    bat_pred = bat_rf.predict(X_test_bat)
    bowl_pred = bowl_ridge.predict(X_test_bowl)

    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.scatter(y_test_bat, bat_pred)
    plt.title("Batting")

    plt.subplot(1,2,2)
    plt.scatter(y_test_bowl, bowl_pred)
    plt.title("Bowling")

    plt.show()


def show_stage_analysis():
    stage_bat = batting_clean.groupby("stage_bucket")["runs"].mean()
    stage_bowl = bowling_clean.groupby("stage_bucket")["economy"].mean()

    fig, ax = plt.subplots(1,2, figsize=(12,5))
    ax[0].bar(stage_bat.index, stage_bat.values)
    ax[0].set_title("Runs by Stage")

    ax[1].bar(stage_bowl.index, stage_bowl.values)
    ax[1].set_title("Economy by Stage")

    plt.show()


def show_player_rankings():
    top_bat = batting_clean.groupby("player")["runs"].mean().sort_values(ascending=False).head(10)
    top_bowl = bowling_clean.groupby("bowler")["economy"].mean().sort_values().head(10)

    fig, ax = plt.subplots(1,2, figsize=(12,6))
    ax[0].barh(top_bat.index, top_bat.values)
    ax[0].set_title("Top Batters")

    ax[1].barh(top_bowl.index, top_bowl.values)
    ax[1].set_title("Top Bowlers")

    plt.show()


# =========================
# PREDICTIONS
# =========================

def predict_new_batting():
    balls = balls_input.value
    sr = sr_input.value

    row = pd.DataFrame([{col:0 for col in bat_features}])
    row["balls"] = balls
    row["strike_rate"] = sr

    pred = bat_rf.predict(row)[0]

    print(f"Predicted Runs: {pred:.2f}")


def predict_new_bowling():
    balls = balls_bowl_input.value
    runs = runs_input.value
    wickets = wickets_input.value

    overs = balls / 6.0
    dot_rate = 0.3
    sr_bpw = balls / wickets if wickets > 0 else np.nan

    row = pd.DataFrame([{col:0 for col in bowl_features}])

    row["balls"] = balls
    row["runs_conceded"] = runs
    row["wickets"] = wickets
    row["overs"] = overs
    row["dot_ball_rate"] = dot_rate
    row["strike_rate_balls_per_wicket"] = sr_bpw

    pred = bowl_ridge.predict(row)[0]

    print(f"Predicted Economy: {pred:.2f}")


# =========================
# UI
# =========================

tabs = widgets.ToggleButtons(
    options=[
        "Model Performance",
        "Actual vs Predicted",
        "Stage Analysis",
        "Player Rankings",
        "Prediction"
    ]
)

output = widgets.Output()

def on_tab_change(change):
    with output:
        clear_output()

        if tabs.value == "Model Performance":
            show_model_performance()

        elif tabs.value == "Actual vs Predicted":
            show_actual_vs_predicted()

        elif tabs.value == "Stage Analysis":
            show_stage_analysis()

        elif tabs.value == "Player Rankings":
            show_player_rankings()

        elif tabs.value == "Prediction":
            display(pred_ui)

tabs.observe(on_tab_change, names="value")

# =========================
# PREDICTION UI
# =========================

mode_toggle = widgets.ToggleButtons(
    options=["Batting", "Bowling"],
    description="Mode:"
)

balls_input = widgets.IntSlider(description="Balls", min=1, max=120, value=20)
sr_input = widgets.FloatText(description="Strike Rate", value=120)

balls_bowl_input = widgets.IntSlider(description="Balls", min=1, max=60, value=24)
runs_input = widgets.IntSlider(description="Runs", min=0, max=100, value=30)
wickets_input = widgets.IntSlider(description="Wickets", min=0, max=10, value=1)

predict_btn = widgets.Button(description="Predict")
pred_output = widgets.Output()

def on_predict(b):
    with pred_output:
        clear_output()
        if mode_toggle.value == "Batting":
            predict_new_batting()
        else:
            predict_new_bowling()

predict_btn.on_click(on_predict)

pred_ui = widgets.VBox([
    mode_toggle,

    widgets.HTML("<b>Batting Inputs</b>"),
    balls_input,
    sr_input,

    widgets.HTML("<b>Bowling Inputs</b>"),
    balls_bowl_input,
    runs_input,
    wickets_input,

    predict_btn,
    pred_output
])

# =========================
# DISPLAY
# =========================
display(tabs, output)

with output:
    show_model_performance()

ToggleButtons(options=('Model Performance', 'Actual vs Predicted', 'Stage Analysis', 'Player Rankings', 'Predi…

Output()